# SupplyMind AI — HistGradientBoosting

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
from supplymind.features.predictions.ml.evaluation import (
    choose_threshold,
    evaluate_probabilities,
    positive_class_probability,
)
from supplymind.features.predictions.ml.preprocessing import build_preprocessor

from supplymind.features.predictions.ml.reporting import (
    save_json,
    save_evaluation_plots,
    save_feature_importance,
)

# -------------------
# Reload evaluation code
# -------------------

import importlib

import supplymind.features.predictions.ml.evaluation as evaluation

evaluation = importlib.reload(evaluation)

positive_class_probability = evaluation.positive_class_probability
choose_threshold = evaluation.choose_threshold
evaluate_probabilities = evaluation.evaluate_probabilities
BinaryMetrics = evaluation.BinaryMetrics

print("Evaluation module:", evaluation.__file__)
print("BinaryMetrics fields:", BinaryMetrics.__annotations__)

from supplymind.features.predictions.ml.training import (
    build_hist_gradient_boosting,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

Evaluation module: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/src/supplymind/features/predictions/ml/evaluation.py
BinaryMetrics fields: {'accuracy': 'float', 'precision': 'float', 'recall': 'float', 'f1': 'float', 'roc_auc': 'float', 'average_precision': 'float', 'balanced_accuracy': 'float', 'specificity': 'float', 'true_negative': 'int', 'false_positive': 'int', 'false_negative': 'int', 'true_positive': 'int', 'false_positive_rate': 'float', 'false_negative_rate': 'float', 'threshold': 'float'}


In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=True,
    sparse_output=False,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_hist_gradient_boosting()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    [
        "balanced_accuracy",
        "f1",
        "recall",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).head(10)

Selected threshold: 0.6200000000000003


,accuracy,precision,recall,f1,roc_auc,average_precision,balanced_accuracy,specificity,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,threshold
42,0.691120,0.881213,0.537290,0.667559,0.739468,0.834411,0.719208,0.901126,8886,975,6229,7233,0.098874,0.462710,0.62
41,0.691120,0.881121,0.537364,0.667590,0.739468,0.834411,0.719194,0.901024,8885,976,6228,7234,0.098976,0.462636,0.61
38,0.691163,0.880857,0.537662,0.667743,0.739468,0.834411,0.719191,0.900720,8882,979,6224,7238,0.099280,0.462338,0.58
39,0.691120,0.880935,0.537513,0.667651,0.739468,0.834411,0.719167,0.900821,8883,978,6226,7236,0.099179,0.462487,0.59
40,0.691077,0.880920,0.537439,0.667589,0.739468,0.834411,0.719130,0.900821,8883,978,6227,7235,0.099179,0.462561,0.60
37,0.691120,0.880472,0.537884,0.667804,0.739468,0.834411,0.719099,0.900314,8878,983,6221,7241,0.099686,0.462116,0.57
43,0.690992,0.881170,0.537067,0.667374,0.739468,0.834411,0.719096,0.901126,8886,975,6232,7230,0.098874,0.462933,0.63
44,0.690906,0.881141,0.536919,0.667251,0.739468,0.834411,0.719022,0.901126,8886,975,6234,7228,0.098874,0.463081,0.64
45,0.690906,0.881141,0.536919,0.667251,0.739468,0.834411,0.719022,0.901126,8886,975,6234,7228,0.098874,0.463081,0.65
46,0.690863,0.881127,0.536844,0.667190,0.739468,0.834411,0.718985,0.901126,8886,975,6235,7227,0.098874,0.463156,0.66


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.6911203532993183,
 'precision': 0.8812134502923976,
 'recall': 0.5372901500519982,
 'f1': 0.6675588371019843,
 'roc_auc': 0.7394683930132029,
 'average_precision': 0.83441126194855,
 'balanced_accuracy': 0.7192078982690779,
 'specificity': 0.9011256464861576,
 'true_negative': 8886,
 'false_positive': 975,
 'false_negative': 6229,
 'true_positive': 7233,
 'false_positive_rate': 0.09887435351384241,
 'false_negative_rate': 0.4627098499480018,
 'threshold': 0.6200000000000003}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "hist_gradient_boosting"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)